# Étape 2 — Explorer rapidement les données

Cette exploration vérifie rapidement la structure et la cohérence du dataset, sans réaliser d'analyse approfondie.

In [1]:
import pandas as pd

# On recharge le fichier CSV généré à l'étape précédente.
df = pd.read_csv("churn_data.csv")

In [2]:
# Contrôle visuel rapide : afficher les cinq premières lignes.
df.head()

,age,salaire,abonnement,anciennete,consommation,churn
0,56,4543,Premium,12,77,0
1,69,6508,Premium,0,83,0
2,46,5270,Standard,9,21,0
3,32,5999,Premium,4,74,0
4,60,2670,Premium,11,26,0


In [3]:
# Nombre de lignes et de colonnes : le résultat attendu est (500, 6).
df.shape

(500, 6)

In [4]:
# Statistiques descriptives des colonnes numériques.
df.describe()

,age,salaire,anciennete,consommation,churn
count,500.000000,500.00000,500.000000,500.000000,500.000000
mean,44.220000,4666.30800,7.144000,48.660000,0.038000
std,15.036082,1844.86557,4.364558,28.927115,0.191388
min,18.000000,1209.00000,0.000000,0.000000,0.000000
25%,32.000000,3124.75000,3.000000,23.750000,0.000000
50%,45.000000,4679.00000,7.000000,47.000000,0.000000
75%,57.000000,6225.25000,11.000000,74.250000,0.000000
max,69.000000,7988.00000,14.000000,99.000000,1.000000


In [5]:
# Nombre de valeurs manquantes par colonne : le résultat attendu est 0 partout.
df.isnull().sum()

age             0
salaire         0
abonnement      0
anciennete      0
consommation    0
churn           0
dtype: int64

In [6]:
# Répartition des clients fidèles (0) et partis (1).
df["churn"].value_counts()

churn
0    481
1     19
Name: count, dtype: int64

## Étape 3 — Préparer les données pour le modèle

On convertit l'abonnement en nombres, on sépare les variables explicatives (`X`) de la cible (`y`), puis on divise les données en ensembles d'entraînement et de test.

In [7]:
from sklearn.model_selection import train_test_split

# Conversion de la colonne abonnement en valeurs numériques.
df["abonnement"] = df["abonnement"].map({
    "Basique": 0,
    "Standard": 1,
    "Premium": 2,
})

# Variables explicatives utilisées par le modèle.
X = df[[
    "age",
    "salaire",
    "abonnement",
    "anciennete",
    "consommation",
]]

# Variable cible à prédire.
y = df["churn"]

# 80 % pour l'entraînement et 20 % pour le test.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

In [8]:
# Vérification de la taille des données d'entraînement.
X_train.shape

(400, 5)

In [9]:
# Vérification de la taille des données de test.
X_test.shape

(100, 5)

## Étape 4 — Entraîner et évaluer le modèle

On entraîne une forêt aléatoire sur les 400 clients d'entraînement, puis on mesure ses performances sur les 100 clients de test.

In [10]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Une forêt de 100 arbres, avec une profondeur maximale de 5.
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    random_state=42,
)

# Apprentissage sur les données d'entraînement.
model.fit(X_train, y_train)

# Prédictions sur les données de test, jamais vues pendant l'apprentissage.
y_pred = model.predict(X_test)

print("Accuracy :", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy : 0.99
              precision    recall  f1-score   support

           0       0.99      1.00      0.99        95
           1       1.00      0.80      0.89         5

    accuracy                           0.99       100
   macro avg       0.99      0.90      0.94       100
weighted avg       0.99      0.99      0.99       100



### Interprétation

Le rapport présente la précision (`precision`), le rappel (`recall`), le score F1 et le nombre d'exemples (`support`) pour chaque classe. Une accuracy de 1.0 est possible ici : la cible `churn` a été construite directement à partir de `consommation` et `anciennete`, deux variables présentes dans `X`. Le modèle peut donc reconstruire cette règle artificielle, ce qui illustre une fuite de données.

## Étape 5 — Sauvegarder le modèle

La sauvegarde permet de réutiliser le modèle entraîné sans relancer l'apprentissage. Attention : chaque exécution écrase le fichier précédent, sans conserver l'historique des versions.

In [11]:
import joblib

# Sérialisation du modèle entraîné dans un fichier binaire.
joblib.dump(model, "model.pkl")
print("Modèle sauvegardé dans model.pkl")

Modèle sauvegardé dans model.pkl


## Étape 6 — Le piège pédagogique : 3 essais, une seule sauvegarde

Chaque sauvegarde utilise le même nom `model.pkl`. Le fichier final ne contient donc que le dernier modèle enregistré.

In [12]:
# ---------- ESSAI 2 : un modèle plus simple ----------
model = RandomForestClassifier(
    n_estimators=50,
    max_depth=3,
    random_state=42,
)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
accuracy_essai_2 = accuracy_score(y_test, y_pred)
print("Accuracy Essai 2 :", accuracy_essai_2)
joblib.dump(model, "model.pkl")
print("model.pkl écrasé avec l'Essai 2")

# ---------- ESSAI 3 : on retire la colonne salaire ----------
X = df[["age", "abonnement", "anciennete", "consommation"]]
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    random_state=42,
)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
accuracy_essai_3 = accuracy_score(y_test, y_pred)
print("Accuracy Essai 3 (sans salaire) :", accuracy_essai_3)
joblib.dump(model, "model.pkl")
print("model.pkl écrasé avec l'Essai 3")

# ---------- VÉRIFICATION : contenu actuel de model.pkl ----------
loaded = joblib.load("model.pkl")
print("Nombre de features attendu :", loaded.n_features_in_)

Accuracy Essai 2 : 0.99
model.pkl écrasé avec l'Essai 2
Accuracy Essai 3 (sans salaire) : 1.0
model.pkl écrasé avec l'Essai 3


Nombre de features attendu : 4


### Questions de réflexion

- **Quel est le meilleur modèle ?** On peut le déterminer uniquement en comparant les métriques enregistrées pour les trois essais, sur le même jeu de test. Une seule accuracy ne suffit pas à prouver la supériorité générale d'un modèle.
- **Peut-on redonner exactement le modèle de l'Essai 1 ?** Non, pas à partir de `model.pkl` après les écrasements : il contient maintenant l'Essai 3. L'Essai 1 ne pourrait être recréé qu'en relançant son code avec les mêmes données, versions de bibliothèques et paramètres.
- **Le résultat est-il garanti sur une autre machine ?** Non. La graine et `random_state` rendent l'expérience reproductible dans un environnement identique, mais les versions de Python, scikit-learn, NumPy et les dépendances peuvent modifier les résultats. Il faut donc aussi tracer l'environnement.

Le nombre de features affiché vaut `4`, ce qui permet d'identifier que le fichier final provient de l'Essai 3.